# Marswalk

Trains a terrain classifier on AI4Mars, then uses it to plan a hazard-aware crew route across Gale Crater.

Written for Colab on a GPU runtime. Checkpoints and outputs are kept on Google Drive so they survive a disconnect.

In [ ]:
!nvidia-smi
!pip install -q segmentation-models-pytorch datasets gradio

## Setup
Mount Drive. `CKPT_DIR` holds the cached arrays, `best.pt` and the exports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = "/content/drive/MyDrive/ai4mars"
os.makedirs(CKPT_DIR, exist_ok=True)

## 1. Data
AI4Mars from the Hugging Face mirror. `test_min3` (labels agreed by at least three experts) is the main held-out test set; `test_min1` is the looser split.

In [ ]:
from datasets import load_dataset

full_train = load_dataset("hassanjbara/AI4MARS", split="train")
test_min3  = load_dataset("hassanjbara/AI4MARS", split="test_min3")   # unanimous expert labels
test_min1  = load_dataset("hassanjbara/AI4MARS", split="test_min1")
print(len(full_train), len(test_min3), len(test_min1))
print(full_train.features)

First version of `prepare()`, superseded by the next cell.

In [ ]:
import numpy as np
IGNORE = 255
CLASSES = ["soil", "bedrock", "sand", "big rock"]

def prepare(row):
    image = np.array(row["image"].convert("RGB"))
    label = np.array(row["label_mask"])
    label = label[..., 0] if label.ndim == 3 else label
    label = label.astype(np.int64)
    for key in ("rover_mask", "range_mask"):
        if row.get(key) is not None:
            m = np.array(row[key])
            m = m[..., 0] if m.ndim == 3 else m
            label[m > 0] = IGNORE
    return image, label

`prepare()` returns `(image, label)`. Pixels under the **rover mask** or **range mask** are set to `255`, which training and every metric ignore. Rows without a label mask return `None`.

In [ ]:
import numpy as np
from PIL import Image
IGNORE = 255

def _as_2d(x):
    if x is None: return None
    a = np.array(x.convert("L")) if isinstance(x, Image.Image) and x.mode not in ("L","P") else np.array(x)
    return a[..., 0] if a.ndim == 3 else a

def prepare(row):
    lab = _as_2d(row.get("label_mask"))
    if lab is None or lab.ndim != 2:
        return None                      # no label mask on this row
    image = np.array(row["image"].convert("RGB"))
    label = lab.astype(np.int64)
    for key in ("rover_mask", "range_mask"):
        m = _as_2d(row.get(key))
        if m is not None and m.ndim == 2:
            label[m > 0] = IGNORE
    return image, label

Only some training rows carry labels.

In [ ]:
try:
    flags = full_train["has_labels"]
    labeled_idx = [i for i, f in enumerate(flags) if f]
except KeyError:
    labeled_idx = [i for i in range(len(full_train)) if full_train[i]["label_mask"] is not None]

print("labeled rows:", len(labeled_idx), "of", len(full_train))
print("first few:", labeled_idx[:5])

Sanity check: labels overlaid on a few frames.
Brown = soil, blue = bedrock, yellow = sand, red = big rock.

In [ ]:
import matplotlib.pyplot as plt
PALETTE = np.array([[150,100,70],[70,110,200],[255,215,0],[220,40,40]], np.uint8)

def overlay(img, lab, alpha=0.55):
    out = img.copy()
    for c in range(4):
        m = lab == c
        out[m] = (1-alpha)*out[m] + alpha*PALETTE[c]
    return out.astype(np.uint8)

for i in labeled_idx[:3]:
    img, lab = prepare(full_train[int(i)])
    print("row", i, "unique label values:", np.unique(lab))
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1); plt.imshow(img); plt.axis("off")
    plt.subplot(1,2,2); plt.imshow(overlay(img, lab)); plt.axis("off")
    plt.show()

## 2. Training set
4,000 labelled rows, resized to 256×256: **bilinear** for images, **nearest** for masks so class IDs never blend. The first 10% become the validation set.

In [ ]:
from PIL import Image
SIZE = 256
N_TRAIN = 4000   # raise to 8000 if the first epoch is fast

flags = full_train["has_labels"]                      # loads the column only, no image decoding
labeled_idx = [i for i, f in enumerate(flags) if f]
print("labeled rows:", len(labeled_idx))

rng = np.random.default_rng(42)
rng.shuffle(labeled_idx)
subset = labeled_idx[:N_TRAIN]

X = np.zeros((len(subset), SIZE, SIZE, 3), np.uint8)
Y = np.full((len(subset), SIZE, SIZE), IGNORE, np.uint8)

for k, i in enumerate(subset):
    img, lab = prepare(full_train[int(i)])
    X[k] = np.array(Image.fromarray(img).resize((SIZE, SIZE), Image.BILINEAR))
    Y[k] = np.array(Image.fromarray(lab.astype(np.uint8)).resize((SIZE, SIZE), Image.NEAREST))
    if k % 500 == 0: print(k, flush=True)

np.save(f"{CKPT_DIR}/X.npy", X); np.save(f"{CKPT_DIR}/Y.npy", Y)

# split by row, 10% validation
n_val = len(X)//10
Xva, Yva, Xtr, Ytr = X[:n_val], Y[:n_val], X[n_val:], Y[n_val:]
print(Xtr.shape, Xva.shape)

Class weights = `1 − class share`, so rare classes count for more in the loss.

In [ ]:
counts = np.bincount(Ytr[Ytr != IGNORE].ravel(), minlength=4)[:4].astype(np.float64)
weights = 1 - counts / counts.sum()
print(dict(zip(CLASSES, counts.astype(int))), weights)

## 3. Baseline
Predict soil everywhere. This is the score the model has to beat.

In [ ]:
def iou_from_cm(cm):
    inter = np.diag(cm).astype(np.float64)
    return inter / (cm.sum(0) + cm.sum(1) - inter + 1e-9)

def cm_all_soil(hf_ds):
    cm = np.zeros((4,4), np.int64)
    for i in range(len(hf_ds)):
        _, lab = prepare(hf_ds[int(i)])
        ok = lab != IGNORE
        if ok.sum() == 0: continue
        cm[:, 0] += np.bincount(lab[ok].ravel(), minlength=4)[:4]   # predict soil everywhere
    return cm

cm_b = cm_all_soil(test_min3)
print("baseline IoU:", dict(zip(CLASSES, iou_from_cm(cm_b).round(3))))
print("baseline mean IoU:", iou_from_cm(cm_b).mean().round(3))
print("baseline pixel accuracy:", (np.diag(cm_b).sum()/cm_b.sum()).round(3))

## 4. Train
DeepLabV3+ / ResNet-50 (ImageNet encoder), weighted cross-entropy, AdamW 3e-4, mixed precision, 6 epochs. Flips are applied to image and mask together. The best validation checkpoint is saved to Drive.

In [ ]:
import torch, torch.nn as nn, segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader

MEAN = np.array([0.485,0.456,0.406], np.float32)
STD  = np.array([0.229,0.224,0.225], np.float32)

class Cached(Dataset):
    def __init__(self, X, Y, train): self.X, self.Y, self.train = X, Y, train
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        img, lab = self.X[i], self.Y[i]
        if self.train:
            if np.random.rand() < 0.5:                      # same flip applied to both
                img, lab = img[:, ::-1], lab[:, ::-1]
            img = np.clip(img.astype(np.float32) * np.random.uniform(0.8, 1.25), 0, 255)
        img = ((img.astype(np.float32)/255.0 - MEAN) / STD).transpose(2,0,1).copy()
        return torch.from_numpy(img), torch.from_numpy(lab.astype(np.int64).copy())

tl = DataLoader(Cached(Xtr, Ytr, True),  batch_size=8, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
vl = DataLoader(Cached(Xva, Yva, False), batch_size=8, shuffle=False, num_workers=2)

model = smp.DeepLabV3Plus("resnet50", encoder_weights="imagenet", classes=4).cuda()
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).cuda(),
                              ignore_index=IGNORE)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda")

@torch.no_grad()
def val_miou():
    model.eval(); cm = np.zeros((4,4), np.int64)
    for x, y in vl:
        with torch.amp.autocast("cuda"):
            p = model(x.cuda()).argmax(1).cpu().numpy()
        y = y.numpy(); ok = y != IGNORE
        cm += np.bincount(4*y[ok] + p[ok], minlength=16).reshape(4,4)
    return iou_from_cm(cm)

EPOCHS, best = 6, -1
for ep in range(EPOCHS):
    model.train()
    for step, (x, y) in enumerate(tl):
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            loss = loss_fn(model(x.cuda()), y.cuda())
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        if step % 100 == 0: print(f"ep{ep} step{step} loss {loss.item():.3f}", flush=True)
    iou = val_miou()
    print(f"epoch {ep}  val mIoU {iou.mean():.3f}  " + str(dict(zip(CLASSES, iou.round(3)))), flush=True)
    if iou.mean() > best:
        best = iou.mean()
        torch.save(model.state_dict(), f"{CKPT_DIR}/best.pt")   # survives a disconnect

## 5. Evaluate
Logits are upsampled to the original label size before `argmax`, so scores are computed against the full-resolution expert masks.

In [ ]:
@torch.no_grad()
def evaluate(hf_ds):
    model.eval(); cm = np.zeros((4,4), np.int64)
    for i in range(len(hf_ds)):
        img, lab = prepare(hf_ds[int(i)])
        H, W = lab.shape
        x = np.array(Image.fromarray(img).resize((SIZE,SIZE), Image.BILINEAR)).astype(np.float32)/255.
        x = torch.from_numpy(((x-MEAN)/STD).transpose(2,0,1)[None]).cuda()
        with torch.amp.autocast("cuda"):
            lg = model(x)
        lg = torch.nn.functional.interpolate(lg.float(), size=(H,W), mode="bilinear", align_corners=False)
        pred = lg.argmax(1)[0].cpu().numpy()
        ok = lab != IGNORE
        if ok.sum() == 0: continue
        cm += np.bincount(4*lab[ok] + pred[ok], minlength=16).reshape(4,4)
    return cm

model.load_state_dict(torch.load(f"{CKPT_DIR}/best.pt"))
for name, ds in [("test_min3", test_min3), ("test_min1", test_min1)]:
    cm = evaluate(ds); iou = iou_from_cm(cm)
    print(name, "| mIoU", round(iou.mean(),3),
          "| per-class", dict(zip(CLASSES, iou.round(3))),
          "| pixel acc", round(np.diag(cm).sum()/cm.sum(), 3))
    print("confusion (rows=true, cols=pred):\n", cm)

## 6. Sand error analysis
Sand is the class the safety argument depends on.
> Needs a `worst` list (test frames ranked by sand error) built in a cell not included here.

In [ ]:
# Phase 6 — Sand error analysis and the overlay (30 min)

# Sand is the class your whole safety argument rests on. Find where the model misses it:

@torch.no_grad()
def predict(img):
    H, W = img.shape[:2]
    x = np.array(Image.fromarray(img).resize((SIZE,SIZE), Image.BILINEAR)).astype(np.float32)/255.
    x = torch.from_numpy(((x-MEAN)/STD).transpose(2,0,1)[None]).cuda()
    with torch.amp.autocast("cuda"):
        lg = model(x)
    lg = torch.nn.functional.interpolate(lg.float(), size=(H,W), mode="bilinear", align_corners=False)
    return lg.argmax(1)[0].cpu().numpy()

missed = 0
for i, r in worst[:6]:
    res = prepare(test_min3[int(i)])
    if res is None: continue
    img, lab = res
    pred = predict(img)
    sand = lab == 2
    n = int(sand.sum()); missed += n
    said = np.bincount(pred[sand].ravel(), minlength=4)[:4]
    ys, _ = np.where(sand)
    print(f"row {i}: {n} sand px ({100*n/lab.size:.2f}% of frame) | "
          f"model said {dict(zip(CLASSES, said.tolist()))} | "
          f"sand spans rows {ys.min()}–{ys.max()} of {lab.shape[0]}")

    plt.figure(figsize=(15,5))
    for k,(t,im) in enumerate([("raw",img),("expert",overlay(img,lab)),("prediction",overlay(img,pred))]):
        plt.subplot(1,3,k+1); plt.imshow(im); plt.title(t); plt.axis("off")
    plt.show()

print(f"\nthese 6 frames hold {missed} sand px = {100*missed/30639905:.3f}% of all sand in test_min3")

Demo inputs for the Gradio app: sand-heavy frames, two general frames and one failure case.

In [ ]:
import os, glob
os.makedirs("demo_imgs", exist_ok=True)

# high-sand frames first (the hero shots)
sand_frames = []
for i in range(len(test_min3)):
    res = prepare(test_min3[i])
    if res is None: continue
    if (res[1] == 2).mean() > 0.25:
        Image.fromarray(res[0]).save(f"demo_imgs/sand_{i}.png")
        sand_frames.append(i)
    if len(sand_frames) >= 3: break

# a couple of general frames + one known failure case
for i in [int(x) for x in [10, 30]] + [worst[0][0]]:
    res = prepare(test_min3[int(i)])
    if res is None: continue
    Image.fromarray(res[0]).save(f"demo_imgs/frame_{i}.png")

print("exported:", sorted(glob.glob("demo_imgs/*.png")))

## 7. Demo
Same interface as the Hugging Face Space. Verdict thresholds on hazard (sand + big rock share): up to 10% traversable, 10–30% caution, over 30% high hazard.

In [ ]:
import gradio as gr, glob

@torch.no_grad()
def segment(pil):
    img = np.array(pil.convert("RGB")); H, W = img.shape[:2]
    x = np.array(Image.fromarray(img).resize((SIZE,SIZE), Image.BILINEAR)).astype(np.float32)/255.
    x = torch.from_numpy(((x-MEAN)/STD).transpose(2,0,1)[None]).cuda()
    lg = torch.nn.functional.interpolate(model(x).float(), size=(H,W), mode="bilinear")
    pred = lg.argmax(1)[0].cpu().numpy()

    pct = {c: float((pred == k).mean()) for k, c in enumerate(CLASSES)}
    hazard = pct["sand"] + pct["big rock"]
    if hazard > 0.30:
        verdict = f"⛔ HIGH HAZARD — {hazard:.0%} of the frame is sand or big rock. Reroute."
    elif hazard > 0.10:
        verdict = f"⚠️ CAUTION — {hazard:.0%} hazardous terrain ahead. Proceed on bedrock where possible."
    else:
        verdict = f"✅ TRAVERSABLE — {hazard:.0%} hazardous terrain."
    return overlay(img, pred), pct, verdict

gr.Interface(
    segment,
    gr.Image(type="pil", label="rover image"),
    [gr.Image(label="terrain  —  brown soil · blue bedrock · yellow sand · red big rock"),
     gr.Label(label="coverage"),
     gr.Textbox(label="traverse assessment")],
    examples=sorted(glob.glob("demo_imgs/*.png"))[:6],
    title="Marswalk — terrain safety from rover imagery",
    description=("DeepLabV3+ (ResNet-50) trained on NASA JPL's AI4Mars dataset. "
                 "0.715 mean IoU, 0.913 IoU on sand on the expert-labeled Curiosity test set "
                 "(test_min3), versus 0.128 for an all-soil baseline. "
                 "Sand is the class that matters most for traverse safety — soft sand is what "
                 "trapped the Spirit rover."),
    flagging_mode="never",
).launch(share=True)

Check the DEM file is on Drive.

In [ ]:
!ls -lh /content/drive/MyDrive/*.tif

## Resume from checkpoint
Start here after a disconnect. Reinstalls packages, redefines the helpers and loads `best.pt`, with no retraining.

In [ ]:
!pip install -q segmentation-models-pytorch datasets gradio

from google.colab import drive
drive.mount('/content/drive')

import numpy as np, torch, torch.nn as nn, os, json
from PIL import Image
import segmentation_models_pytorch as smp
from datasets import load_dataset

CKPT_DIR = "/content/drive/MyDrive/ai4mars"
SIZE, IGNORE = 256, 255
CLASSES = ["soil", "bedrock", "sand", "big rock"]
MEAN = np.array([0.485,0.456,0.406], np.float32)
STD  = np.array([0.229,0.224,0.225], np.float32)
PALETTE = np.array([[150,100,70],[70,110,200],[255,215,0],[220,40,40]], np.uint8)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

def _as_2d(x):
    if x is None: return None
    a = np.array(x.convert("L")) if isinstance(x, Image.Image) and x.mode not in ("L","P") else np.array(x)
    return a[..., 0] if a.ndim == 3 else a

def prepare(row):
    lab = _as_2d(row.get("label_mask"))
    if lab is None or lab.ndim != 2: return None
    image = np.array(row["image"].convert("RGB"))
    label = lab.astype(np.int64)
    for key in ("rover_mask", "range_mask"):
        m = _as_2d(row.get(key))
        if m is not None and m.ndim == 2:
            label[m > 0] = IGNORE
    return image, label

def overlay(img, lab, alpha=0.55):
    out = img.copy()
    for c in range(4):
        m = lab == c
        out[m] = (1-alpha)*out[m] + alpha*PALETTE[c]
    return out.astype(np.uint8)

model = smp.DeepLabV3Plus("resnet50", encoder_weights=None, classes=4).to(DEV)
model.load_state_dict(torch.load(f"{CKPT_DIR}/best.pt", map_location=DEV))
model.eval()
print("checkpoint loaded")

@torch.no_grad()
def predict(img):
    H, W = img.shape[:2]
    x = np.array(Image.fromarray(img).resize((SIZE,SIZE), Image.BILINEAR)).astype(np.float32)/255.
    x = torch.from_numpy(((x-MEAN)/STD).transpose(2,0,1)[None]).to(DEV)
    lg = model(x).float()
    lg = torch.nn.functional.interpolate(lg, size=(H,W), mode="bilinear", align_corners=False)
    return lg.argmax(1)[0].cpu().numpy()

test_min3 = load_dataset("hassanjbara/AI4MARS", split="test_min3")
print("ready:", len(test_min3), "test rows")

## 8. Exports
The checkpoint plus 8 test frames spanning the hazard range (raw and classified) for the site.

In [ ]:
# The steps
# Step 1 — Export everything from Colab (30 min, do now)

# Before the runtime disconnects and you lose an hour retraining. Run this and download the zip:

import os, json, zipfile
os.makedirs("export/frames", exist_ok=True)

# save the checkpoint for the HF Space
torch.save(model.state_dict(), "export/best.pt")

# pick 8 frames spanning the hazard range
chosen, records = [], []
for i in range(len(test_min3)):
    res = prepare(test_min3[i])
    if res is None: continue
    img, lab = res
    pred = predict(img)
    pct = {c: float((pred == k).mean()) for k, c in enumerate(CLASSES)}
    hazard = pct["sand"] + pct["big rock"]
    records.append((i, hazard, pct))

records.sort(key=lambda r: r[1])
picks = [records[0], records[len(records)//6], records[len(records)//3],
         records[len(records)//2], records[2*len(records)//3],
         records[5*len(records)//6], records[-2], records[-1]]

meta = []
for n, (i, hazard, pct) in enumerate(picks):
    img, lab = prepare(test_min3[int(i)])
    pred = predict(img)
    Image.fromarray(img).save(f"export/frames/wp{n}_raw.png")
    Image.fromarray(overlay(img, pred)).save(f"export/frames/wp{n}_seg.png")
    meta.append({"id": n, "source_row": int(i), "coverage": pct,
                 "hazard": round(hazard, 4)})

json.dump(meta, open("export/frames.json", "w"), indent=2)
!cd export && zip -qr ../marswalk_export.zip . && echo done
from google.colab import files; files.download("marswalk_export.zip")

In [ ]:
from google.colab import files
files.download(f"{CKPT_DIR}/best.pt")

Six demo inputs, named by predicted hazard share.

In [ ]:
import os, zipfile
os.makedirs("examples", exist_ok=True)

records = []
for i in range(len(test_min3)):
    res = prepare(test_min3[i])
    if res is None: continue
    img, _ = res
    pred = predict(img)
    hazard = float(((pred == 2) | (pred == 3)).mean())
    records.append((i, hazard))

records.sort(key=lambda r: r[1])
picks = [records[1], records[len(records)//5], records[len(records)//2],
         records[3*len(records)//5], records[-6], records[-2]]

for n, (i, h) in enumerate(picks):
    img, _ = prepare(test_min3[int(i)])
    name = f"examples/{n+1}_hazard{int(round(h*100)):02d}pct.png"
    Image.fromarray(img).save(name)
    print(name)

!zip -qr examples.zip examples
from google.colab import files; files.download("examples.zip")

Three-panel figures: rover image, expert labels, prediction.

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("figs", exist_ok=True)
for n, (i, h) in enumerate(picks):
    img, lab = prepare(test_min3[int(i)])
    pred = predict(img)
    fig, ax = plt.subplots(1, 3, figsize=(15,5))
    for a,(t,im) in zip(ax, [("rover image",img),("expert labels",overlay(img,lab)),("Marswalk prediction",overlay(img,pred))]):
        a.imshow(im); a.set_title(t); a.axis("off")
    plt.tight_layout(); plt.savefig(f"figs/fig{n+1}.png", dpi=130, bbox_inches="tight"); plt.close()
!zip -qr figs.zip figs
files.download("figs.zip")

## 9. Terrain
The MOLA/HRSC blended DEM at 200 m/px. The next two cells downsample the whole global map and are superseded by the windowed crop further down.

In [ ]:
!pip install -q rasterio

In [ ]:
import rasterio, numpy as np
from rasterio.enums import Resampling

DEM_PATH = "/content/drive/MyDrive/Mars_MOLA_blend200ppx_HRSC_DEM_clon0dd_200mpp_lzw.tif"
MAXD = 400

with rasterio.open(DEM_PATH) as src:
    print("full size:", src.shape, "dtype:", src.dtypes[0])
    f = max(1, max(src.shape) // MAXD)
    out_h, out_w = src.height // f, src.width // f
    dem = src.read(1, out_shape=(1, out_h, out_w),
                   resampling=Resampling.average).astype(np.float32)   # float32, not 64
    transform = src.transform * src.transform.scale(src.width/out_w, src.height/out_h)
    crs, px_m = src.crs, abs(src.transform.a) * f

dem = np.where(dem < -1e5, np.nan, dem)
dem = np.nan_to_num(dem, nan=float(np.nanmedian(dem)))
H, W = dem.shape

gy, gx = np.gradient(dem.astype(np.float64), px_m)
slope_deg = np.degrees(np.arctan(np.hypot(gx, gy)))
print(f"grid {H}x{W}, {px_m:.1f} m/px, elev {dem.min():.0f}–{dem.max():.0f} m, slope max {slope_deg.max():.1f}°")

Free the training arrays before loading the DEM (Colab RAM).

In [ ]:
import gc
for n in ["X","Y","Xtr","Ytr","Xva","Yva","full_train","cands","records"]:
    if n in globals(): del globals()[n]
gc.collect()

In [ ]:
import rasterio, numpy as np, gc
from rasterio.enums import Resampling

DEM_PATH = "/content/drive/MyDrive/Mars_MOLA_blend200ppx_HRSC_DEM_clon0dd_200mpp_lzw.tif"
MAXD = 200                     # was 400

with rasterio.open(DEM_PATH) as src:
    print("full size:", src.shape, "dtype:", src.dtypes[0])
    f = max(1, max(src.shape) // MAXD)
    out_h, out_w = src.height // f, src.width // f
    dem = src.read(1, out_shape=(1, out_h, out_w),
                   resampling=Resampling.average).astype(np.float32)
    transform = src.transform * src.transform.scale(src.width/out_w, src.height/out_h)
    crs, px_m = src.crs, abs(src.transform.a) * f

dem = np.where(dem < -1e5, np.nan, dem)
dem = np.nan_to_num(dem, nan=float(np.nanmedian(dem)))
H, W = dem.shape
gc.collect()

gy, gx = np.gradient(dem.astype(np.float64), px_m)
slope_deg = np.degrees(np.arctan(np.hypot(gx, gy)))
print(f"grid {H}x{W}, {px_m:.1f} m/px, elev {dem.min():.0f}-{dem.max():.0f} m, slope max {slope_deg.max():.1f}°")

**Version used.** Crop ±0.35° around Gale Crater, convert degrees to metres on the Mars sphere, then compute slope.

In [ ]:
import rasterio, numpy as np, gc
from rasterio.enums import Resampling
from rasterio.windows import from_bounds

DEM_PATH = "/content/drive/MyDrive/Mars_MOLA_blend200ppx_HRSC_DEM_clon0dd_200mpp_lzw.tif"
LON, LAT, HALF = 137.4, -4.5, 0.35      # Gale Crater, ~±0.35° ≈ 20 km
MAXD = 300

with rasterio.open(DEM_PATH) as src:
    win = from_bounds(LON-HALF, LAT-HALF, LON+HALF, LAT+HALF, src.transform)
    f = max(1, int(max(win.height, win.width) // MAXD))
    out_h, out_w = max(8, int(win.height)//f), max(8, int(win.width)//f)
    dem = src.read(1, window=win, out_shape=(1, out_h, out_w),
                   resampling=Resampling.average).astype(np.float32)
    wt = src.window_transform(win)
    transform = wt * wt.scale(win.width/out_w, win.height/out_h)
    crs = src.crs

dem = np.where(dem < -1e5, np.nan, dem)
dem = np.nan_to_num(dem, nan=float(np.nanmedian(dem)))
H, W = dem.shape

# transform is in DEGREES, so convert to metres on the Mars sphere
R = 3396190.0
deg_lat_m = np.pi * R / 180.0
deg_lon_m = deg_lat_m * np.cos(np.radians(LAT))
dy_m, dx_m = abs(transform.e) * deg_lat_m, abs(transform.a) * deg_lon_m
px_m = (dy_m + dx_m) / 2
gc.collect()

gy, gx = np.gradient(dem.astype(np.float64), dy_m, dx_m)
slope_deg = np.degrees(np.arctan(np.hypot(gx, gy)))
print(f"grid {H}x{W}, {px_m:.0f} m/px, elev {dem.min():.0f}-{dem.max():.0f} m, "
      f"slope max {slope_deg.max():.1f}°, mean {slope_deg.mean():.1f}°")

In [ ]:
print(H, W, round(px_m), len(test_min3))   # expect 207 207 200 322

## 10. Route planning
**Imaging stations.** AI4Mars frames carry no coordinates, so 12 real test frames that span the hazard range are placed at random grid positions (fixed seed). Station hazard = share of the frame predicted as sand or big rock.

In [ ]:
import json, gc
from rasterio.transform import xy

N_STATIONS = 12
rng = np.random.default_rng(7)

cands = []
for i in range(len(test_min3)):
    res = prepare(test_min3[i])
    if res is None: continue
    img, _ = res
    pred = predict(img)
    cands.append({"row": int(i),
                  "hazard": float(((pred == 2) | (pred == 3)).mean()),
                  "coverage": {c: round(float((pred == k).mean()), 4) for k, c in enumerate(CLASSES)}})
    del img, pred
    if len(cands) >= 60: break
gc.collect()

cands.sort(key=lambda c: c["hazard"])
sel = [cands[int(x)] for x in np.linspace(3, len(cands)-4, N_STATIONS).astype(int)]

stations = []
for n, c in enumerate(sel):
    r = int(rng.integers(H//8, 7*H//8)); cc = int(rng.integers(W//8, 7*W//8))
    lon, lat = xy(transform, r, cc)
    stations.append({"id": n, "rc": [r, cc], "lon": float(lon), "lat": float(lat),
                     "elev_m": float(dem[r, cc]), "slope_deg": float(slope_deg[r, cc]), **c})
print("stations:", len(stations), "| hazard range:",
      round(min(s['hazard'] for s in stations),3), "-", round(max(s['hazard'] for s in stations),3))

**Cost surface.** Station hazard is spread with a Gaussian kernel (σ ≈ 15 cells ≈ 3 km), then
`cost = 1 + 4·(slope/15°)² + 6·hazard`. Cells steeper than 15° are impassable.

In [ ]:
SIGMA_PX = max(H, W) / 14.0
yy, xx = np.mgrid[0:H, 0:W]

hazard = np.zeros((H, W)); wsum = np.zeros((H, W)) + 1e-9
for s in stations:
    r, c = s["rc"]
    k = np.exp(-(((yy-r)**2 + (xx-c)**2) / (2*SIGMA_PX**2)))
    hazard += k * s["hazard"]; wsum += k
hazard /= wsum

SLOPE_LIMIT, K_SLOPE, K_HAZ = 15.0, 4.0, 6.0
slope_n = np.clip(slope_deg / SLOPE_LIMIT, 0, 1)
cost = 1.0 + K_SLOPE * slope_n**2 + K_HAZ * hazard
cost[slope_deg > SLOPE_LIMIT] = np.inf
print(f"impassable cells: {100*np.isinf(cost).mean():.1f}%")

**A\*** on the 8-connected grid with an octile heuristic, compared against a straight line between the same endpoints. Paths are de-duplicated before they are measured.

In [ ]:
import heapq

def astar(cost, start, goal):
    H, W = cost.shape
    nbrs = [(-1,0,1),(1,0,1),(0,-1,1),(0,1,1),(-1,-1,2**.5),(-1,1,2**.5),(1,-1,2**.5),(1,1,2**.5)]
    def h(a):
        dy, dx = abs(a[0]-goal[0]), abs(a[1]-goal[1])
        return max(dy,dx) + (2**.5 - 1)*min(dy,dx)
    g = {start: 0.0}; came = {}; pq = [(h(start), start)]; seen = set()
    while pq:
        _, cur = heapq.heappop(pq)
        if cur == goal:
            p = [cur]
            while p[-1] in came: p.append(came[p[-1]])
            return p[::-1]
        if cur in seen: continue
        seen.add(cur)
        for dy, dx, step in nbrs:
            nb = (cur[0]+dy, cur[1]+dx)
            if not (0 <= nb[0] < H and 0 <= nb[1] < W) or nb in seen: continue
            c = cost[nb]
            if not np.isfinite(c): continue
            ng = g[cur] + step * c
            if ng < g.get(nb, np.inf):
                g[nb] = ng; came[nb] = cur
                heapq.heappush(pq, (ng + h(nb), nb))
    return None

def ok(p):
    r = int(np.clip(p[0], 1, H-2)); c = int(np.clip(p[1], 1, W-2))
    if np.isfinite(cost[r, c]): return (r, c)
    fin = np.argwhere(np.isfinite(cost))
    return tuple(int(v) for v in fin[np.argmin(((fin - [r, c])**2).sum(1))])

def line(a, b, n=400):
    return [(int(round(a[0]+(b[0]-a[0])*t)), int(round(a[1]+(b[1]-a[1])*t)))
            for t in np.linspace(0, 1, n)]

def dedupe(p):                      # FIX: collapse repeated cells before measuring
    out = [p[0]]
    for q in p[1:]:
        if q != out[-1]: out.append(q)
    return out

def score(p):
    if not p: return None
    p = dedupe(p)                   # FIX: always measure a deduped path
    d = sum(px_m*(2**.5 if abs(p[i][0]-p[i-1][0])+abs(p[i][1]-p[i-1][1])==2 else 1)
            for i in range(1, len(p)))
    s = np.array([slope_deg[r,c] for r,c in p]); z = np.array([hazard[r,c] for r,c in p])
    return {"distance_m": round(d), "max_slope_deg": round(float(s.max()),1),
            "mean_slope_deg": round(float(s.mean()),1),
            "mean_hazard": round(float(z.mean()),3),
            "pct_over_limit": round(100*float((s > SLOPE_LIMIT).mean()),1)}

START = ok((int(H*0.85), int(W*0.15)))
GOAL  = ok((int(H*0.15), int(W*0.85)))
path   = dedupe(astar(cost, START, GOAL))
direct = dedupe(line(START, GOAL))

ds, dp = score(direct), score(path)
print("route cells:", len(path))
print("straight line :", ds)
print("Marswalk route:", dp)
print(f"\n--> extra distance: {dp['distance_m']-ds['distance_m']:+,} m "
      f"({100*(dp['distance_m']/ds['distance_m']-1):+.0f}%)")
print(f"--> hazard exposure: {ds['mean_hazard']:.3f} -> {dp['mean_hazard']:.3f} "
      f"({100*(1-dp['mean_hazard']/ds['mean_hazard']):.0f}% reduction)")

Write `site.json` for the web page, plus an elevation / slope / cost figure with both routes.

In [ ]:
import matplotlib.pyplot as plt, os
os.makedirs("figs", exist_ok=True)
lonlat = lambda rc: [float(v) for v in xy(transform, rc[0], rc[1])]

site = {
  "site": {"name": "Gale Crater - lower Mount Sharp",
           "objective": "Characterize boxwork formations",
           "center": [LON, LAT], "px_m": round(px_m,1), "grid": [H, W]},
  "route": {"coords": [lonlat(rc) for rc in path[::3]], "stats": score(path)},
  "direct": {"coords": [lonlat(START), lonlat(GOAL)], "stats": score(direct)},
  "stations": [{k: s[k] for k in ("id","lon","lat","elev_m","slope_deg","hazard","coverage","row")}
               for s in stations],
  "model": {"miou": 0.715, "sand_iou": 0.913, "baseline_miou": 0.128,
            "pixel_acc": 0.961, "test_split": "AI4Mars test_min3 (expert, unanimous)"},
  "params": {"slope_limit_deg": SLOPE_LIMIT, "k_slope": K_SLOPE, "k_hazard": K_HAZ},
  "note": ("Elevation and slope are real MOLA/HRSC data for Gale Crater. AI4Mars frames "
           "carry no rover coordinates, so imaging stations are placed across the survey "
           "area and each is assigned a real Curiosity frame whose terrain composition the "
           "model measured. Station placement is illustrative; the terrain classification "
           "and all reported metrics are not.")
}
json.dump(site, open("figs/site.json","w"), indent=2)

fig, ax = plt.subplots(1, 3, figsize=(18,5))
ax[0].imshow(dem, cmap="terrain"); ax[0].set_title("MOLA/HRSC elevation")
ax[1].imshow(slope_deg, cmap="magma", vmax=20); ax[1].set_title("slope (deg)")
ax[2].imshow(np.where(np.isinf(cost), np.nan, cost), cmap="viridis"); ax[2].set_title("traversal cost")
pr, pd_ = np.array(path), np.array(direct)
ax[2].plot(pd_[:,1], pd_[:,0], "w--", lw=1.5, label="direct")
ax[2].plot(pr[:,1], pr[:,0], "r-", lw=2.5, label="Marswalk")
ax[2].scatter([s["rc"][1] for s in stations], [s["rc"][0] for s in stations], c="cyan", s=30, ec="k")
ax[2].legend()
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig("figs/route.png", dpi=140, bbox_inches="tight"); plt.show()

!cp figs/site.json figs/route.png /content/drive/MyDrive/ai4mars/

Station frames (raw and classified) for the site.

In [ ]:
import os
os.makedirs("figs", exist_ok=True)
for s in stations:
    img, _ = prepare(test_min3[s["row"]])
    pred = predict(img)
    Image.fromarray(img).save(f"figs/st{s['id']}_raw.png")
    Image.fromarray(overlay(img, pred)).save(f"figs/st{s['id']}_seg.png")
    del img, pred
!cd figs && zip -qr ../marswalk_assets.zip . && cp ../marswalk_assets.zip /content/drive/MyDrive/ai4mars/
from google.colab import files; files.download("marswalk_assets.zip")

## 11. 3D view
Rebuilds the DEM, hazard, cost and route directly from `site.json`, so the 3D view matches the live site exactly.

In [ ]:
!pip install -q rasterio plotly

import json, numpy as np, rasterio, heapq
from rasterio.enums import Resampling
from rasterio.windows import from_bounds
from rasterio.transform import rowcol, xy

DEM_PATH = "/content/drive/MyDrive/Mars_MOLA_blend200ppx_HRSC_DEM_clon0dd_200mpp_lzw.tif"
S = json.load(open("/content/drive/MyDrive/ai4mars/site.json"))
LON, LAT = S["site"]["center"]
HALF, MAXD = 0.35, 300

with rasterio.open(DEM_PATH) as src:
    win = from_bounds(LON-HALF, LAT-HALF, LON+HALF, LAT+HALF, src.transform)
    f = max(1, int(max(win.height, win.width) // MAXD))
    out_h, out_w = max(8, int(win.height)//f), max(8, int(win.width)//f)
    dem = src.read(1, window=win, out_shape=(1, out_h, out_w),
                   resampling=Resampling.average).astype(np.float32)
    wt = src.window_transform(win)
    transform = wt * wt.scale(win.width/out_w, win.height/out_h)

dem = np.nan_to_num(np.where(dem < -1e5, np.nan, dem), nan=float(np.nanmedian(dem)))
H, W = dem.shape
R = 3396190.0
deg_lat_m = np.pi * R / 180.0
dy_m, dx_m = abs(transform.e)*deg_lat_m, abs(transform.a)*deg_lat_m*np.cos(np.radians(LAT))
px_m = (dy_m + dx_m) / 2
gy, gx = np.gradient(dem.astype(np.float64), dy_m, dx_m)
slope_deg = np.degrees(np.arctan(np.hypot(gx, gy)))

# stations straight from site.json — identical to the live site
stations = []
for s in S["stations"]:
    r, c = rowcol(transform, s["lon"], s["lat"])
    stations.append({**s, "rc": [int(np.clip(r,0,H-1)), int(np.clip(c,0,W-1))]})

SIGMA_PX = max(H, W) / 14.0
yy, xx = np.mgrid[0:H, 0:W]
hazard = np.zeros((H, W)); wsum = np.zeros((H, W)) + 1e-9
for s in stations:
    r, c = s["rc"]
    k = np.exp(-(((yy-r)**2 + (xx-c)**2) / (2*SIGMA_PX**2)))
    hazard += k * s["hazard"]; wsum += k
hazard /= wsum

P = S["params"]
SLOPE_LIMIT, K_SLOPE, K_HAZ = P["slope_limit_deg"], P["k_slope"], P["k_hazard"]
cost = 1.0 + K_SLOPE*np.clip(slope_deg/SLOPE_LIMIT,0,1)**2 + K_HAZ*hazard
cost[slope_deg > SLOPE_LIMIT] = np.inf

def astar(cost, start, goal):
    Hh, Ww = cost.shape
    nbrs = [(-1,0,1),(1,0,1),(0,-1,1),(0,1,1),(-1,-1,2**.5),(-1,1,2**.5),(1,-1,2**.5),(1,1,2**.5)]
    def h(a):
        dy, dx = abs(a[0]-goal[0]), abs(a[1]-goal[1])
        return max(dy,dx) + (2**.5-1)*min(dy,dx)
    g = {start:0.0}; came={}; pq=[(h(start),start)]; seen=set()
    while pq:
        _, cur = heapq.heappop(pq)
        if cur == goal:
            p=[cur]
            while p[-1] in came: p.append(came[p[-1]])
            return p[::-1]
        if cur in seen: continue
        seen.add(cur)
        for dy,dx,st in nbrs:
            nb=(cur[0]+dy,cur[1]+dx)
            if not (0<=nb[0]<Hh and 0<=nb[1]<Ww) or nb in seen: continue
            cc=cost[nb]
            if not np.isfinite(cc): continue
            ng=g[cur]+st*cc
            if ng<g.get(nb,np.inf):
                g[nb]=ng; came[nb]=cur; heapq.heappush(pq,(ng+h(nb),nb))
    return None

def ok(p):
    r=int(np.clip(p[0],1,H-2)); c=int(np.clip(p[1],1,W-2))
    if np.isfinite(cost[r,c]): return (r,c)
    fin=np.argwhere(np.isfinite(cost))
    return tuple(int(v) for v in fin[np.argmin(((fin-[r,c])**2).sum(1))])

def dedupe(p):
    out=[p[0]]
    for q in p[1:]:
        if q!=out[-1]: out.append(q)
    return out

START = ok(rowcol(transform, *S["direct"]["coords"][0]))
GOAL  = ok(rowcol(transform, *S["direct"]["coords"][1]))
path   = dedupe(astar(cost, START, GOAL))
direct = dedupe([(int(round(START[0]+(GOAL[0]-START[0])*t)),
                  int(round(START[1]+(GOAL[1]-START[1])*t))) for t in np.linspace(0,1,400)])

print(H, W, round(px_m), len(path), len(direct), len(stations))
print("impassable:", f"{100*np.isinf(cost).mean():.1f}%")

Interactive Plotly surface coloured by traversal cost (impassable cells show as holes), with ×4 vertical exaggeration. Saved as `marswalk_3d.html`.

In [ ]:
!pip install -q plotly
import plotly.graph_objects as go
import numpy as np

EXAG = 4                                   # vertical exaggeration
x_km = np.arange(W) * px_m / 1000
y_km = np.arange(H) * px_m / 1000
z = dem * EXAG
csurf = np.where(np.isinf(cost), np.nan, cost)   # impassable renders as holes

fig = go.Figure()

fig.add_surface(
    x=x_km, y=y_km, z=z, surfacecolor=csurf,
    colorscale=[[0,"#1a2b4a"],[0.4,"#4a3d2a"],[0.7,"#b06a30"],[1,"#ff7a2f"]],
    colorbar=dict(title="traversal<br>cost", tickfont=dict(color="#9b8e82")),
    lighting=dict(ambient=0.55, diffuse=0.85, roughness=0.9),
    name="terrain")

def line3d(cells, color, width, label, lift=120):
    r = np.array([c[0] for c in cells]); c = np.array([c[1] for c in cells])
    return go.Scatter3d(
        x=c*px_m/1000, y=r*px_m/1000, z=dem[r,c]*EXAG + lift,
        mode="lines", line=dict(color=color, width=width), name=label)

fig.add_trace(line3d(direct, "#ffffff", 3, "direct route"))
fig.add_trace(line3d(path,   "#ff3b30", 7, "MARSWALK route"))

sr = np.array([s["rc"][0] for s in stations]); sc = np.array([s["rc"][1] for s in stations])
fig.add_trace(go.Scatter3d(
    x=sc*px_m/1000, y=sr*px_m/1000, z=dem[sr,sc]*EXAG + 160,
    mode="markers", name="imaging stations",
    marker=dict(size=5, color=[s["hazard"] for s in stations],
                colorscale=[[0,"#52e08a"],[0.3,"#ffb020"],[1,"#ff5252"]],
                line=dict(color="#070605", width=1)),
    text=[f"ST-{s['id']:02d} · {s['hazard']*100:.0f}% hazard" for s in stations],
    hoverinfo="text"))

fig.update_layout(
    title=dict(text="MARSWALK — Gale Crater traverse (vertical exaggeration ×4)",
               font=dict(color="#f6efe6", size=16)),
    paper_bgcolor="#070605", font=dict(family="Barlow", color="#9b8e82"),
    scene=dict(
        xaxis=dict(title="km", backgroundcolor="#070605", gridcolor="#2a221c", color="#9b8e82"),
        yaxis=dict(title="km", backgroundcolor="#070605", gridcolor="#2a221c", color="#9b8e82"),
        zaxis=dict(title="elevation ×4", backgroundcolor="#070605", gridcolor="#2a221c", color="#9b8e82"),
        aspectratio=dict(x=1, y=1, z=0.45),
        camera=dict(eye=dict(x=1.5, y=-1.5, z=0.9))),
    margin=dict(l=0, r=0, t=46, b=0), height=720)

fig.write_html("marswalk_3d.html", include_plotlyjs="cdn")
fig.show()

In [ ]:
!cp marswalk_3d.html /content/drive/MyDrive/ai4mars/
!ls -lh marswalk_3d.html
from google.colab import files; files.download("marswalk_3d.html")